<h1 style="font-size: 36px; color: blue;">STOWA Proevenverzameling tool v5.0</h1>

Deze Jupyter notebook bevat functies voor het opstellen van lokale of regionale proevenverzamelingen voor het bepalen van geotechnische parameters. De methode is ontwikkeld voor het uitvoeren van analyses in relatie tot de geotechnische stabiliteit van dijken, maar kan ook breder worden toegepast. De notebook dient tevens als handleiding om de gebruiker stapsgewijs te ondersteunen bij het opstellen van een proevenverzameling. De onderliggende pythoncodes zijn opgenomen in de map ../pv_tool. Zie Readme.md voor meer informatie over de installatie en het gebruik van de onderliggende code zonder gebruik van Jupyter notebook. 

Met de beschikbare functies kunnen zowel gedraineerde als ongedraineerde sterkteparameters worden berekend alsmede enkele samendrukkingsparameters. Van deze parameters worden verwachtingswaarde, karakteristieke waarde en rekenwaarde bepaald.

De functies zijn opgesteld conform de werkwijze beschreven in [Statistische methoden t.b.v. proevenverzamelingen, DIV, v1.0], zie tevens: https://publicwiki.deltares.nl/spaces/HWBPMacro/pages/217120830/Sterkte+van+grond#Sterktevangrond-150. De tool bevat tevens enkele hulpmiddelen voor het onderscheiden van groepen in een verzameling op basis van verschillende kenmerken.

De onderliggende data om een proevenverzameling samen te stellen is beschreven in een vaste structuur. Deze structuur is vastgelegd in een uitwisselformat. Het uitwisselformat-database-proevenverzameling_versie_4_2x.xlsx. De geotechnische laboratoria kennen deze database en kunnen deze database vullen met resultaten van grond- en laboratoriumonderzoek. Op deze wijze ontstaat er uniformering op het gebied van data-uitwisseling en –opslag van proefresultaten.

NB. De verantwoordelijkheid voor het gebruik van deze tools ligt bij de gebruiker.

## Inhoudsopgave

- [Stap 1: Opgeven benodigde data en export locatie](#Stap-1:-Opgeven-benodigde-data-en-export-locatie)
- [Stap 2: Importeren van data](#Stap-2:-Importeren-van-data)
- [Stap 3: Kies materiaalfactor en alpha](#Stap-3:-Instellingen-(materiaalfactor-en-alpha))
- [Stap 4: C-Phi analyse](#Stap-4:-Bepalen-gedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-fit-(C-en-Phi)-of-obv-schematiseringshandleiding-(Phi))
- [Stap 5: SHANSEP](#Stap-5:-Bepalen-ongedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-methode-Shansep-(S,-m-en-POP))
- [Stap 6: SU-Tabel](#Stap-6:-Bepalen-ongedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-Su-tabel-methode)
- [Extra: Figuur genereren](#Extra:-Genereren-van-een-figuur)

## Benodigde installaties

Voor het werken met de PV-tool dient het benodigd package te worden geinstalleerd. Hierin staat de achterliggende code om deze tool werkbaar te maken. 

Ook dienen verschillende python-packages te worden geinstalleerd. Zie tevens README.md en requirements.txt.
Na de installatie dient de kernel handmatig opnieuw opgestart te worden via het menu Kernel -> Restart.

In [14]:
# Importeren van benodigde packages, geef het correcte pad op bij path_to_wheel =
# Na het eerte keer installeren van de benodigde packages de kernel handmatig opnieuw optarten via het menu Kernel → Restart

path_to_wheel = r"C:\.......\pv-tool\dist\pv_tool-0.3.4-py3-none-any.whl"
!pip install "{path_to_wheel}"

In [1]:
# all imports - ALTIJD UITVOEREN
import importlib.util
import ipywidgets as widgets
from IPython.display import display, Markdown
from ipyfilechooser import FileChooser
from pathlib import Path
import os
import plotly.io as pio
pio.renderers.default = "notebook_connected"

from pv_tool.imports.import_data import Dbase
from pv_tool.cphi_analysis.c_phi_analysis import CPhiAnalyse
from pv_tool.cphi_analysis.variables import *

# widgetfuncties nog verder aanvullen!
from pv_tool.utilities.widget_functions_cphi import *
from pv_tool.utilities.widget_functions_shansep import *
from pv_tool.utilities.widget_functions_su import *

## Stap 1: Opgeven benodigde data en export locatie
De benodigde data voor het bepalen van geotechnische parameters kan op drie manieren in deze notebook worden ingeladen:

1. Op basis van de Excel‑template voor de Proevenverzameling 5.0.
Deze template bevat alle data conform het Uitwisselformat 4.2, aangevuld met de benodigde invoer- en uitvoerdata voor deze tooling.

2. Op basis van het Excelbestand Uitwisselformat-database-proevenverzameling_versie_4_2x.xlsx.
In dit bestand zijn nog geen proevenverzamelingen onderscheiden.
Zie ook: https://github.com/kkpdata/Proevenverzamelingentool/

3. Op basis van de Excelversie van de Proevenverzamelingtool (vanaf versie 4.2n of hoger).
In deze versie zijn reeds proevenverzamelingen onderscheiden en zijn geotechnische parameters vastgesteld.
Let op: de onderscheiden verzamelingen worden als uitgangspunt ingeladen, maar de sterkteparameters worden opnieuw berekend. Onderliggende keuzes — zoals het gewenste rekpercentage en de geselecteerde raaklijnen — worden niet uit de Excelversie ingelezen.

Na het inladen wordt de Proevenverzamelingtool of het Uitwisselformaat automatisch omgezet naar het template van Proevenverzamelingtool 5.0.
Na deze conversie dient met dit nieuwe bestand verder te worden gewerkt.

In [2]:
import_dropdown, import_filechooser, export_dirchooser, export_namebox = setup_interactive_import_export()

Dropdown(description='Template:', index=2, layout=Layout(width='400px'), options=('Proevenverzamelingtool 4.2n…

FileChooser(path='C:\', filename='', title='Selecteer een bestand om te uploaden', show_hidden=False, select_d…

Output()

## Stap 2: Importeren van data

Voor het importeren van data zijn twee opties beschikbaar.

---

### 2a. Importeren zonder validatie
Deze optie is uitsluitend bedoeld voor het inladen van: **Proevenverzamelingtool_5.0.xlsx**.
De data wordt hierbij direct ingelezen. Deze functie mag alleen gebruikt worden indien er reeds een gevalideerde dataset beschikbaar is. Na het aanbrengen van wijzigingen in de data altijd stap 2b toepassen voor het importeren. Dit is noodzakelijk voor het herberekenen van bepaalde parameters benodigde voor het correct uitvoeren van de functies bij aanpassingen aan de data.

---

### 2b. Importeren, (her)berekenen parameters analysekolommen, valideren en exporteren

Gebruik deze optie voor:
- Nieuwe datasets op basis van Uitwisselformat-database-proevenverzameling_versie_4_2x.xlsx
- De Excelversie van de Proevenverzamelingtool (vanaf versie 4.2n of hoger)
- Alle situaties waarin de proefdata wordt aangepast of aangevuld, onafhankelijk van het format

---

### Stap 2a: Importeren data zonder validatie of herberekening analysekolommen

### LET OP! Aleen mogelijk voor de proevenverzamelingtool_5.0.xlsx

In [3]:
dbase = Dbase()
handle_import_only(
    dbase,
    import_dropdown,
    import_filechooser,
    process_import_only
)

**Template-code:** Dbase

**Data geïmporteerd uit:** C:\python_cursus\PVtool2025_github\PV-tool\import_files\WSRL 2025 PVtool5_0_gevalideerd_v6.xlsx

### Stap 2b: Importeren, (her)berekenen parameters analysekolommen, valideren en exporteren
Substappen binnen 2b:

1. **Importeren en (her)berekenen van analysekolommen**: De benodigde analysekolommen worden opnieuw afgeleid uit de data.
2. **Valideren van de data**: De dataset wordt voor de relevante parameters gecontroleerd op volledigheid, consistentie en juistheid.
3. **Exporteren conform Template Proevenverzamelingtool_5.0.xlsx**: De gevalideerde dataset wordt weggeschreven naar het standaard formaat waarmee verder wordt gewerkt.

---

### Stap 2b.1. Importeren en (her)berekenen van parameters en analysekolommen

De importfunctie leest de dataset in en vult vervolgens een reeks parameters aan of berekent deze opnieuw (bij gebruik van *Proevenverzamelingtool 5.0*). Deze parameters worden als extra kolommen aan het dataframe toegevoegd.

#### Toegevoegde of herberekende kolommen
- Toekennen van een proef aan een specifieke verzameling: PV_NAAM, PV_OPMERKING  
- Parameters consolidatie en spanningstoestand: ANA_TERREINSPANNING, ANA_TXT_MAX_VERTICALE_CONSOLIDATIE_SPANNING, ANA_DSS_MAX_CONSOLIDATIE_SPANNING, ANA_TXT_CONSOLIDATIE_TYPE_VOORSTEL, ANA_TXT_CONSOLIDATIE_TYPE_HANDMATIG, ANA_TXT_CONSOLIDATIE_TYPE_REKEN, ANA_DSS_CONSOLIDATIE_TYPE_VOORSTEL, ANA_DSS_CONSOLIDATIE_TYPE_HANDMATIG ANA_DSS_CONSOLIDATIE_TYPE_REKEN, ANA_GRENSSPANNING_PROEF, ANA_POP_VELD, ANA_POP_VELD_GEMIDDELD, ANA_GRENSSPANNING_VOORSTEL, ANA_GRENSSPANNING_HANDMATIG ANA_GRENSSPANNING_REKEN, OCR_TXT, OCR_DSS.

---

Deze kolommen zijn nodig om vast te stellen of een triaxiaal- of DSS‑proef is uitgevoerd op een normaal geconsolideerd monster **(NC; OCR = 1)**, of  
een overgeconsolideerd monster **(OC; OCR > 1)**. Daarnaast is een (schatting van de) **grensspanning, POP en OCR** benodigd

#### Toelichting bij ANA_kolommen voor bepaling CONSOLIDATIE_TYPE
De tooling doet hiervoor automatisch een voorstel in .._CONSOLIDATIE_TYPE_VOORSTEL, gebaseerd op:
- vergelijking van de **consolidatiespanning** (σ'vc) met de **terreinspanning** (σ'vi)
- indien σ'vc niet meer dan **±30%** afwijkt van σ'vi → proef wordt als **OC** voorgesteld  
- bij grotere afwijkingen → proef wordt als **NC** voorgesteld

> **Belangrijk:** Het voorgestelde consolidatietype moet altijd worden gecontroleerd.  
> In de praktijk ligt een OC‑proef meestal dicht bij de terreinspanning, terwijl een NC‑proef daar ruim boven ligt — maar uitzonderingen komen voor.

Een gebruiker kan een handmatige waarde invoeren in:
- **ANA_TXT_CONSOLIDATIE_TYPE_HANDMATIG** (triaxiaal)
- **ANA_DSS_CONSOLIDATIE_TYPE_HANDMATIG** (DSS)

#### Toelichting bij ANA_kolommen voor bepaling grensspanning, POP en OCR
Voor het bepalen van POP en OCR is een grensspanning nodig. De tooling doet hiervoor automatisch een voorstel in ANA_GRENSSPANNING_VOORSTEL. De grensspanning en hieruitvolgende POP en OCR wordt bepaald volgens de volgende logica:
1. Indien op dezelfde regel een grensspanning uit een samendrukkingsproef of CRS‑proef beschikbaar is wordt deze gebruikt.
2. Indien deze ontbreekt, wordt een voorstel gedaan op basis van de gemiddelde POP per boring (POP = grensspanning − effectieve terreinspanning)
3. Indien binnen een boring geen enkele grensspanning beschikbaar is, wordt geen waarde ingevuld. In dat geval kan geen automatische OCR‑schatting worden gemaakt.

> **Belangrijk:** De voorgestelde grensspanning en hieruitvolgende POP en OCR moet altijd worden gecontroleerd.  

Een gebruiker kan een handmatige waarde invoeren in:
- **ANA_GRENSSPANNING_HANDMATIG**

---
NB. Bij gebruik van reeds bestaande Excelversie van de Proevenverzamelingtool (vanaf versie 4.2n of hoger) is de kolom ANA_GRENSSPANNING_HANDMATIG vaak al ingevuld.In dat geval worden deze waarden **overgenomen** door de importfunctie.

---

### Stap 2b.2. Validatie van de data
De validatie bestaat uit een reeks controles waarbij per regel en per proefsoort wordt bekeken: Is er een proef uitgevoerd? (d.w.z. zijn er relevante waarden ingevuld?) en zijn alle verplichte velden benodigd voor de PV‑tool volledig en correct ingevuld?

#### Resultaten van de validatie
De uitkomsten van de validatie worden opgeslagen in twee Excel‑bestanden. In deze bestanden wordt per gevalideerde kolom en per regel aangegeven wat het resultaat is. Daarnaast wordt per kolom weergegeven hoeveel fouten zijn aangetroffen. Er wordt onderscheid gemaakt tussen:
- **Critical errors**  Fouten of ontbrekende waarden die de berekening van parameters direct beïnvloeden. Deze moeten worden gecorrigeerd of aangevuld voordat verder kan worden gewerkt.
- **Warnings**  Aanbevelingen of mogelijke aandachtspunten. Deze hoeven niet noodzakelijkerwijs te worden aangepast, maar verdienen controle.

Beide bestanden worden opgeslagen in dezelfde directory als de gebruikte importmap.

In [4]:
#Stap 2b.1 en 2b.2
dbase = Dbase()
handle_import_export(
    dbase,
    import_dropdown,
    import_filechooser,
    export_dirchooser,
    process_import_and_validate
)

Er is geen bestand geselecteerd. Selecteer een bestand voordat je verder gaat.


### Stap 2b.3: Na oplossen errors exporteren data naar template proevenverzamelingtool 5.0

Indien de validatie goed is doorlopen en afgerond worden proevenverzamelingtool (vanaf versie 4.2n of hoger) of Uitwisselformat-database-proevenverzameling_versie_4_2x.xlsx weggeschreven naar het Template_PVtool5_0.xlsx. Indien reeds gebruik is gemaakt van het nieuwe template worden de analyse kolommen opnieuw berekend. Dit is noodzakelijk indien er data gewijzigd is.

Indien er reeds een gevalideerde dataset beschikbaar is conform het Template_PVtool5_0 laadt de data dan in via stap 2a. Dit kost aanzienlijk minder rekentijd bij een grote dataset.

In [97]:
# #Stap 2b.3 export dbase-template
if import_dropdown.value == "Proevenverzamelingtool 5.0":
    print('Export gelijk aan import')
else:
    process_export(
        dbase,
        export_dir=Path(export_dirchooser.selected_path),
        filename=export_namebox.value if export_namebox.value else None
    )

Export gelijk aan import


## Stap 3: Instellingen (materiaalfactor en alpha)

Kies materiaalfactor en alpha op basis van de type verzameling. Deze instelling wordt gebruikt voor alle analyses.

In [4]:
# Stel materiaalfactor en alpha in
alpha_widget, partphi_widget, partcoh_widget = toon_grid_settings()

**Pas alpha aan (lokaal = 1,0, regionaal = 0,75):**

**Pas materiaalfactoren aan:**

## Stap 4: Bepalen gedraineerde parameters triaxiaalproeven en DSS-proeven obv fit (C en Phi) of obv schematiseringshandleiding (Phi)

- [Terug naar boven](#Inhoudsopgave)
- [Stap 2: Importeren van data](#Stap-2:-Importeren-van-data)

In deze stap worden op basis van de data opgenomen in Proevenverzameling_5.0.xlsx gedraineerde parameters bepaald. 

Kies eerst de verzameling waarop de statistische analyse wordt uitgevoerd. Als door de gebruiker nog geen aparte verzamelingen zijn onderscheiden, worden alle triaxiaalproeven samengevoegd onder TXT en alle DSS‑proeven onder DSS. De gebruiker kan de indeling en naamgeving handmatig aanpassen in Excel in het tabblad **\[Dbase5_0\]**, kolom **PV_NAAM**.

Vervolgens worden de analyse‑instellingen opgegeven:
- Proeftype + Analysemethode: *TXT* voor triaxiaalproeven, *DSS* voor DSS‑proeven, *CPhi* voor Mohr‑Coulomb‑parameters (c en φ) en *SH* voor CSSM‑parameters volgens de schematiseringshandleiding macrostabiliteit (φ)
- Rekpercentage voor aflezen van s’ en t: 2%, 5%, 15%, eindrek of pieksterkte

Indien eerder parameters zijn vastgesteld, worden de gekozen rekpercentages en raaklijnen automatisch ingeladen.

In de grafieken kunnen aanvullende verzamelingen ter vergelijking worden weergegeven. **Let op:** deze extra verzamelingen worden niet meegenomen in de statistische analyse.

Punten in de grafieken zijn gelabeld met **ALG__BORING_MONSTERNR_ID** (eerste kolom in \[Dbase5_0\]).

Deze tooling bevat geen functies om groepen automatisch te onderscheiden. Met behulp van [Extra: Figuur genereren](#Extra:-Genereren-van-een-figuur) kan de gebruiker zelf aanvullende grafieken maken om mogelijke groeperingen te onderzoeken.

In [5]:
(dropdown_type_proef, dropdown_verzameling, dropdown_rekpercentage_txt, 
 dropdown_rekpercentage_dss, container_rekpercentage, output_rekpercentage, 
 multi_select_verzameling, gekozen_rekpercentage) = dropdown_widgets(dbase)

**Kies type proef:**

Dropdown(description='Type proef:', layout=Layout(width='400px'), options=('TXT_CPhi', 'DSS_CPhi', 'TXT_SH', '…

**Kies verzameling voor statistische analyse:**

Dropdown(description='Verzameling:', layout=Layout(width='400px'), options=('geen', 'TXT_testset_klei', 'TXT',…

**Kies rekpercentage s'en t:**

Output()

Output()

**Kies één of meerdere verzamelingen om naast de gekozen verzameling voor de statistische analyse te tonen:**

SelectMultiple(description='Vergelijk met:', index=(0,), layout=Layout(height='150px', width='400px'), options…

In [6]:
# run analyse op basis van de gekozen verzameling
analyse, coh_gem, phi_kar, coh_kar = voer_cphi_analyse_uit(
    dbase=dbase,
    import_dropdown=import_dropdown,
    import_filechooser=import_filechooser,
    dropdown_verzameling=dropdown_verzameling,
    dropdown_type_proef=dropdown_type_proef,
    dropdown_rekpercentage_txt=dropdown_rekpercentage_txt,
    dropdown_rekpercentage_dss=dropdown_rekpercentage_dss,
    export_dir_widget=export_dirchooser,
    export_name_widget=export_namebox,
    gekozen_rekpercentage=gekozen_rekpercentage,
    toon_cphi_tabel=toon_cphi_tabel,
    partphi_widget=partphi_widget,
    partcoh_widget=partcoh_widget,
    alpha_widget=alpha_widget
)

Data na filtering: 65 rijen gevonden


C:\python_cursus\PVtool2025_github\PV-tool\pv_tool\cphi_analysis\calc_parameters.py:214: UserWarning:

Om berekening standaarddeviatie voor D-stability te kunnen doen moet rekenwaarde van de cohesie waarde positief zijn, gevonden waarde: -2.552. Rekenwaarde cohesie wordt aangepast naar 0.01 om berekening standaarddeviatie voor D-stability te kunnen doen



Er zijn geen eerdere resultaten gevonden voor de opgegeven parameters.


**Stel de raaklijnen voor de gemiddelde en karakteristieke waarden van de cohesie en hoek van inwendige wrijving vast:**

Op basis van regressie wordt een eerste benadering gegeven voor het snijpunt met de y-as (a1) en de helling (a2) voor de gemiddelde en karakteristieke waarde.

Toets in de volgende stap bij het genereren van de grafieken of de raaklijnen juist zijn gekozen en pas deze aan naar eigen inzicht

De invoer wordt automatisch opgehaald uit het template_PVtool5_0.xlsx [resultaten] indien er eerder resultaten zijn opgeslagen voor de betreffende verzameling.

**Verzameling en rekpercentage: TXT_klei_14_16, s'-t bij: ['eindsterkte']**

**Opgeven invoer effectieve schuifsterkteparameters (fit):**

GridspecLayout(children=(Label(value='Beschrijving', layout=Layout(grid_area='widget001')), Label(value='Benad…

In [8]:
# Voer C-Phi analyse uit met gekozen invoer en toon grafieken. 
analyse, output_df = show_cphi_analysis(
    dbase,
    dropdown_verzameling,
    dropdown_type_proef,
    dropdown_rekpercentage_txt,
    dropdown_rekpercentage_dss,
    coh_gem,
    phi_kar,
    coh_kar,
    multi_select_verzameling,
    alpha_widget,
    partphi_widget,
    partcoh_widget
)

Data na filtering: 65 rijen gevonden
Data na filtering: 65 rijen gevonden
                              tan phi [-]  phi [graden]  cohesie [kPa]
Verwachtingswaarde               0.628717     32.158254       7.087330
Karakteristieke waarde           0.625002     32.005455       0.117925
Rekenwaarde                      0.625002     32.005455       0.117925
Standaarddeviatie D-stability         [-]      0.093035      27.023396
Data na filtering: 65 rijen gevonden
fysisch realiseerbare ondergrens gebaseerd op phi kar handmatig en cohesie kar handmatig
gemiddelde gebaseerd op helling gecorrigeerd en cohesie gem handmatig


Data na filtering: 65 rijen gevonden
fysisch realiseerbare ondergrens gebaseerd op phi kar handmatig en cohesie kar handmatig
gemiddelde gebaseerd op helling gecorrigeerd en cohesie gem handmatig


#### Resultaten opslaan
Met de opslagfunctie worden de gekozen invoer en de berekende gedraineerde sterkteparameters vastgelegd in **Proevenverzameling_5.0.xlsx** in het tabblad **\[Resultaten c‑phi\]**. Per regel zijn resultaten vastgelegd met onderscheid tussen het type proef (TXT of DSS), analysetype (c–φ of φ volgens SH macrostabiliteit) en gekozen rekpercentage. Elke combinatie kan afzonderlijk worden opgeslagen. Aan elke opslag wordt automatisch een *timestamp* toegevoegd, zodat eerdere resultaten behouden blijven.

---

#### Factsheets en figuren
Er wordt een factsheet gegenereerd met alle relevante invoer, uitvoer en de gegenereerde grafieken. Deze wordt opgeslagen als pdf bestand. Daarnaast worden de grafieken afzonderlijk weggeschreven als interactieve HTML‑bestanden. Punten in de grafieken zijn gelabeld met **ALG__BORING_MONSTERNR_ID**. (dit is de eerste kolom in **\[Dbase5_0\]**).

In [9]:
# Resultaten opslaan in Proevenverzameling_5.0.xlsx tabblad [Resultaten c‑phi], genereren factsheets en figuren
_ = export_results(
    analyse,
    dropdown_verzameling,
    dropdown_type_proef,
    import_dropdown,
    import_filechooser,
    export_dir_widget=export_dirchooser,
    export_name_widget=export_namebox
)

Resultaat toegevoegd aan template in tabblad 'Resultaten c-phi'.
figuur opgeslagen als HTML: C:\python_cursus\PVtool2025_github\PV-tool\import_files\c-phi_analyse_TXT_klei_14_16.html
Data na filtering: 65 rijen gevonden
PDF succesvol opgeslagen op: C:\python_cursus\PVtool2025_github\PV-tool\import_files/c_phi_pdf_export_TXT_klei_14_16_TXT_CPhi_15procent_rek.pdf


## Stap 5: Bepalen ongedraineerde parameters triaxiaalproeven en DSS-proeven obv methode Shansep (S, m en POP)

- [Terug naar boven](#Inhoudsopgave)
- [Stap 2: Importeren van data](#Stap-2:-Importeren-van-data)

In deze stap worden op basis van de data opgenomen in Proevenverzameling_5.0.xlsx ongedraineerde Shansep parameters bepaald. 

Kies eerst de verzameling waarop de statistische analyse wordt uitgevoerd. Als door de gebruiker nog geen aparte verzamelingen zijn onderscheiden, worden alle triaxiaalproeven samengevoegd onder TXT en alle DSS‑proeven onder DSS. De gebruiker kan de indeling en naamgeving handmatig aanpassen in Excel in het tabblad **\[Dbase5_0\]**, kolom **PV_NAAM**.

Vervolgens worden de analyse‑instellingen opgegeven:
- Proeftype + Analysemethode: *TXT* voor triaxiaalproeven, *DSS* voor DSS‑proeven, *S_POP* voor het bepalen van de Shansep parameters
- Rekpercentage voor aflezen van Su: 2%, 5%, 15%, eindrek of pieksterkte

Indien eerder parameters zijn vastgesteld, worden de gekozen rekpercentages en raaklijnen automatisch ingeladen.

In de grafieken kunnen aanvullende verzamelingen ter vergelijking worden weergegeven. **Let op:** deze extra verzamelingen worden niet meegenomen in de statistische analyse.

Punten in de grafieken zijn gelabeld met **ALG__BORING_MONSTERNR_ID** (eerste kolom in \[Dbase5_0\]).

Deze tooling bevat geen functies om groepen automatisch te onderscheiden. Met behulp van [Extra: Figuur genereren](#Extra:-Genereren-van-een-figuur) kan de gebruiker zelf aanvullende grafieken maken om mogelijke groeperingen te onderzoeken.

In [10]:
(dropdown_type_proef_shansep, dropdown_verzameling_shansep, dropdown_rekpercentage_txt_shansep, 
 dropdown_rekpercentage_dss_shansep, container_rekpercentage_shansep, output_rekpercentage_shansep,
 multi_select_verzameling_shansep, gekozen_rekpercentage_shansep) = dropdown_widgets_shansep(dbase)

**Kies type proef:**

Dropdown(description='Type proef:', layout=Layout(width='400px'), options=('TXT_S_POP', 'DSS_S_POP'), value='T…

**Kies verzameling voor statistische analyse:**

Dropdown(description='Verzameling:', layout=Layout(width='400px'), options=('geen', 'TXT_testset_klei', 'TXT',…

**Kies rekpercentage Su:**

Output()

Output()

**Kies één of meerdere verzamelingen om naast de gekozen verzameling voor de statistische analyse te tonen:**

SelectMultiple(description='Vergelijk met:', index=(0,), layout=Layout(height='150px', width='400px'), options…

In [11]:
# Run analyse en show grid
analyse, df_gem, df_kar, widgets_gem, widgets_kar = run_shansep_analysis(
        dbase,
        dropdown_verzameling_shansep,
        alpha_widget,
        import_dropdown,
        import_filechooser,
        dropdown_rekpercentage_txt_shansep,
        dropdown_rekpercentage_dss_shansep,
        dropdown_type_proef_shansep,
        export_dir_widget=None,
        export_name_widget=None
)

**Stel de raaklijnen vast voor de bepaling van de gemiddelde en karakteristieke waarden van:**

 - S en POP door het opgeven van het snijpunt met de y-as (a1) en de schuifsterkteratio (a2)
- S en m door het opgeven van het schuifsterkterratio (a1) en de macht (a2)

Op basis van regressie wordt een eerste benadering gegeven voor de gemiddelde en karakteristieke waarden van al deze parameters, tevens wordt de POP op basis van de CRS of samendrukkingsproeven gegeven en de S op basis van de normaal geconsolideerde proeven.

Toets in de volgende stap bij het genereren van de grafieken of de raaklijnen juist zijn gekozen en pas deze aan naar eigen inzicht.
De invoer wordt automatisch opgehaald uit het `template_PVtool5_0.xlsx` ([resultaten Shansep]) indien er eerder resultaten zijn opgeslagen voor de betreffende verzameling.
    

**Opgeven gemiddelde waarden (fit):**

GridspecLayout(children=(Label(value='Parameters', layout=Layout(grid_area='widget001', width='300px')), Label…

**Opgeven karakteristieke waarden (fit):**

GridspecLayout(children=(Label(value='Parameters', layout=Layout(grid_area='widget001', width='300px')), Label…

> **LET OP:** De in regel 1 en 2 voorgestelde waarden voor het snijpunt met de y‑as, de normaal geconsolideerde schuifsterkteratio,
> de sterktetoename‑exponent *m* en de daaruit volgende POP zijn het resultaat van een automatische fit op de beschikbare datapunten. Het is aan de gebruiker om te beoordelen of deze resulteren in realistische Shansep‑parameters en of deze consistent zijn met elkaar en met:
>
> - de gemiddelde en karakteristieke POP‑waarde binnen de verzameling op basis van (ANA_GRENSSPANNING_REKEN - ANA_TERREINSPANNING);
> - de gemiddelde en karakteristieke waarde van S, bepaald uit de NC‑proeven in de verzameling.
>
> Een goede statistische fit betekent **niet automatisch** dat de Shansep‑parameters correct zijn.  
> Dit risico is groter bij **kleine datasets** of datasets met **grote spreiding** (mogelijk gevolg van een onjuiste groepering).

In [12]:
# Voer SHANSEP-analyse uit met invoer en toon grafieken
analyse, output_df = show_shansep_analysis(
    dbase,
    widgets_kar,
    widgets_gem,
    dropdown_verzameling_shansep,
    dropdown_type_proef_shansep,
    dropdown_rekpercentage_txt_shansep,
    dropdown_rekpercentage_dss_shansep,
    multi_select_verzameling_shansep,
    alpha_widget
)

                           S [-]     m [-]  POP [kPa]
Verwachtingswaarde      0.310000  0.750000  36.559140
Karakteristieke waarde  0.275000  0.700000  23.376623
Standaarddeviatie       0.022128  0.031081   9.380111


#### Resultaten opslaan
Met de opslagfunctie worden de gekozen invoer en de berekende ongedraineerde sterkteparameters vastgelegd in **Proevenverzameling_5.0.xlsx** in het tabblad **\[Resultaten SHANSEP\]**. Per regel zijn resultaten vastgelegd met onderscheid tussen het type proef (TXT of DSS) en gekozen rekpercentage. Elke combinatie kan afzonderlijk worden opgeslagen. Aan elke opslag wordt automatisch een *timestamp* toegevoegd, zodat eerdere resultaten behouden blijven.

---

#### Factsheets en figuren
Er wordt een factsheet gegenereerd met alle relevante invoer, uitvoer en de gegenereerde grafieken. Deze wordt opgeslagen als pdf bestand. Daarnaast worden de grafieken afzonderlijk weggeschreven als interactieve HTML‑bestanden. Punten in de grafieken zijn gelabeld met **ALG__BORING_MONSTERNR_ID**. (dit is de eerste kolom in **\[Dbase5_0\]**).

In [150]:
# Exports
_ = export_shansep_results(
    analyse,
    dropdown_verzameling_shansep,
    dropdown_type_proef_shansep,
    import_dropdown,
    import_filechooser,
    export_dir_widget=export_dirchooser,
    export_name_widget=export_namebox)

Tabblad resultaten SHANSEP in dbase excel bestaat al en wordt aangevuld
Resultaat toegevoegd aan template in tabblad 'Resultaten SHANSEP'.
Figuren opgeslagen als HTML in : C:\python_cursus\PVtool2025_github\PV-tool\import_files
SHANSEP PDF export voltooid: C:\python_cursus\PVtool2025_github\PV-tool\import_files/shansep_pdf_export_TXT_klei_17.5_21_diep_TXT_S_POP_15procent_rek.pdf


## Stap 6: Bepalen ongedraineerde parameters triaxiaalproeven en DSS-proeven obv Su tabel methode

- [Terug naar boven](#Inhoudsopgave)
- [Stap 2: Importeren van data](#Stap-2:-Importeren-van-data)

In deze stap wordt op basis van de data opgenomen in Proevenverzameling_5.0.xlsx een Su tabel opgesteld. 

Kies eerst de verzameling waarop de statistische analyse wordt uitgevoerd. Als door de gebruiker nog geen aparte verzamelingen zijn onderscheiden, worden alle triaxiaalproeven samengevoegd onder TXT en alle DSS‑proeven onder DSS. De gebruiker kan de indeling en naamgeving handmatig aanpassen in Excel in het tabblad **\[Dbase5_0\]**, kolom **PV_NAAM**.

Vervolgens worden de analyse‑instellingen opgegeven:
- Proeftype + Analysemethode: *TXT* voor triaxiaalproeven, *DSS* voor DSS‑proeven, *Su_tabel* voor het bepalen van de Su tabel
- Rekpercentage voor aflezen van Su: 2%, 5%, 15%, eindrek of pieksterkte

Indien eerder parameters zijn vastgesteld, worden de gekozen rekpercentages en raaklijnen automatisch ingeladen.

In de grafieken kunnen aanvullende verzamelingen ter vergelijking worden weergegeven. **Let op:** deze extra verzamelingen worden niet meegenomen in de statistische analyse.

Punten in de grafieken zijn gelabeld met **ALG__BORING_MONSTERNR_ID** (eerste kolom in \[Dbase5_0\]).

Deze tooling bevat geen functies om groepen automatisch te onderscheiden. Met behulp van [Extra: Figuur genereren](#Extra:-Genereren-van-een-figuur) kan de gebruiker zelf aanvullende grafieken maken om mogelijke groeperingen te onderzoeken.

In [13]:
(dropdown_type_proef_su, dropdown_verzameling_su, dropdown_rekpercentage_txt_su,
 dropdown_rekpercentage_dss_su, container_rekpercentage_su, output_rekpercentage_su,
 multi_select_verzameling_su, gekozen_rekpercentage_su) = dropdown_widgets_su(dbase)

**Kies type proef:**

Dropdown(description='Type proef:', layout=Layout(width='400px'), options=('TXT_su_tabel', 'DSS_su_tabel'), va…

**Kies verzameling voor statistische analyse:**

Dropdown(description='Verzameling:', layout=Layout(width='400px'), options=('geen', 'TXT_testset_klei', 'TXT',…

**Kies rekpercentage Su:**

Output()

Output()

**Kies één of meerdere verzamelingen om naast de gekozen verzameling voor de statistische analyse te tonen:**

SelectMultiple(description='Vergelijk met:', index=(0,), layout=Layout(height='150px', width='400px'), options…

In [19]:
# run analyse en show grid
analyse, handmatige_widgets = run_su_analysis(
    dbase,
    dropdown_verzameling_su,
    alpha_widget,
    import_dropdown,
    import_filechooser,
    dropdown_rekpercentage_txt_su,
    dropdown_rekpercentage_dss_su,
    dropdown_type_proef_su,
    export_dir_widget=None,
    export_name_widget=None)

**Opgegeven karakteristieke waarden (fit):**

GridspecLayout(children=(Label(value='Parameters', layout=Layout(grid_area='widget001', width='200px')), HTML(…

In [20]:
# show results
analyse, output_df = show_su_analysis(dbase,
        dropdown_verzameling_su,
        dropdown_type_proef_su,
        dropdown_rekpercentage_txt_su,
        dropdown_rekpercentage_dss_su,
        handmatige_widgets[0].value,
        handmatige_widgets[1].value,
        handmatige_widgets[2].value,
        multi_select_verzameling_su,
        alpha_widget,
        import_dropdown,
        import_filechooser                      
)

                        svgm [kPa]     m [-]  vc_fit [-]
Verwachtingswaarde       12.951284  0.646054    0.100000
Karakteristieke waarde   10.277942      0.63    0.100000
Standaarddeviatie         0.211007       [-]    0.099751


#### Resultaten opslaan
Met de opslagfunctie worden de gekozen invoer en de berekende ongedraineerde sterkteparameters vastgelegd in **Proevenverzameling_5.0.xlsx** in het tabblad **\[Resultaten SU-tabel-m\]**. Per regel zijn resultaten vastgelegd met onderscheid tussen het type proef (TXT of DSS) en gekozen rekpercentage. Elke combinatie kan afzonderlijk worden opgeslagen. Aan elke opslag wordt automatisch een *timestamp* toegevoegd, zodat eerdere resultaten behouden blijven.

---

#### Factsheets en figuren
Er wordt een factsheet gegenereerd met alle relevante invoer, uitvoer en de gegenereerde grafieken. Deze wordt opgeslagen als pdf bestand. Daarnaast worden de grafieken afzonderlijk weggeschreven als interactieve HTML‑bestanden. Punten in de grafieken zijn gelabeld met **ALG__BORING_MONSTERNR_ID**. (dit is de eerste kolom in **\[Dbase5_0\]**).

In [161]:
# Resultaten opslaan in Proevenverzameling_5.0.xlsx tabblad [Resultaten SU-tabel-m], genereren factsheets en figuren
_ = export_su_results(
    analyse,
    dropdown_verzameling_su,
    dropdown_type_proef_su,
    import_dropdown,
    import_filechooser,
    export_dir_widget=export_dirchooser,
    export_name_widget=export_namebox
)

Tabblad Resultaten SU-tabel - m in dbase excel bestaat nog niet en wordt aangemaakt
Resultaat toegevoegd aan template in tabblad 'Resultaten SU-tabel-m'.
Figuren opgeslagen als HTML in: C:\python_cursus\PVtool2025_github\PV-tool\import_files
Sutabel PDF export voltooid: C:\python_cursus\PVtool2025_github\PV-tool\import_files/sutabel_pdf_export_TXT_klei_17.5_21_ondiep_TXT_su_tabel_15procent_rek.pdf


## Extra: Genereren van extra figuren
De gebruiker kan hier zelf grafieken maken. Er zijn een aantal voorbeelden opgenomen.

- [Terug naar inhoudsopgave](#Inhoudsopgave)
- [Stap 4: C-Phi analyse](#Stap-4:-Bepalen-gedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-fit-(C-en-Phi)-of-obv-schematiseringshandleiding-(Phi))
- [Stap 5: SHANSEP](#Stap-5:-Bepalen-ongedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-methode-Shansep-(S,-m-en-POP))
- [Stap 6: SU-Tabel](#Stap-6:-Bepalen-ongedraineerde-parameters-triaxiaalproeven-en-DSS-proeven-obv-Su-tabel-methode)

In [21]:
# Zelf genereren figuren: Droog volumegewicht - monsterniveau

import plotly.express as px

import_data = dbase.dbase_df

filtered_data = import_data[
    (import_data['ALG_OPDRACHTGEVER'] == 'WSRL') & #regels filteren op een bepaalde waarde
    (~import_data['PV_NAAM'].fillna('').str.contains('afgekeurd|uitbijter|onbetrouwbaar|telaag|DSS', case=False, na=False)) #regels uitsluiten
]

# Droog volumegewicht - diepte 
fig1 = px.scatter(
    filtered_data,
    y='TXT_SS_MONSTERNIVEAU',
    x='TXT_SS_VOLUMEGEWICHT_DRG',
    color='PV_NAAM',
    title='Droog volumegewicht - monsterniveau',
    labels={
        'TXT_SS_MONSTERNIVEAU': 'Monsterniveau [NAP m]',
        'TXT_SS_VOLUMEGEWICHT_DRG': 'Droog volumegewicht [kN/m3]',
        'PV_NAAM': 'PV naam'
    },
    template='plotly_white',
    opacity=0.8,
    hover_data=['ALG__REGEL']  # voeg label uit DB toe aan tooltip
)

fig1.update_traces(marker=dict(size=7))   # pas marker-grootte aan
fig1.show()



In [22]:
# Zelf genereren figuren: Su tabel met onderscheid in droog volumegewicht

import plotly.express as px
import plotly.graph_objects as go

import_data = dbase.dbase_df

filtered_data3 = import_data[
    (import_data['ALG_OPDRACHTGEVER'] == 'WSRL') &  # regels filteren op een bepaalde waarde
    (import_data['ANA_TXT_CONSOLIDATIE_TYPE_REKEN'] == 'OC') &  # regels filteren op een bepaalde waarde
    (
        (import_data['PV_NAAM'] == 'TXT_klei_12_14') |
        (import_data['PV_NAAM'] == 'TXT_klei_14_16') |
        (import_data['PV_NAAM'] == 'TXT_klei_16_17.5') |
        (import_data['PV_NAAM'] == 'TXT_klei_17.5_21_ondiep') |
        (import_data['PV_NAAM'] == 'TXT_klei_17.5_21_diep')
    )
]

# Droog volumegewicht - diepte 
fig3 = px.scatter(
    filtered_data3,
    y='TXT_SS_T_15%',
    x='ANA_TXT_MAX_VERTICALE_CONSOLIDATIE_SPANNING',
    color='TXT_SS_VOLUMEGEWICHT_DRG',
    title='Su tabel bij 15% rek',
    labels={
        'TXT_SS_MONSTERNIVEAU': 'Monsterniveau [NAP m]',
        'TXT_SS_VOLUMEGEWICHT_DRG': 'Droog volumegewicht [kN/m3]',
        'PV_NAAM': 'PV naam'
    },
    template='plotly_white',
    opacity=0.8,
    hover_data=['ALG__REGEL']  # voeg label uit DB toe aan tooltip
)

fig3.update_xaxes(range=[0, None])
fig3.update_yaxes(range=[0, None])
fig3.show()



In [23]:
import plotly.express as px
import plotly.graph_objects as go

import_data = dbase.dbase_df

filtered_data4 = import_data[
    (import_data['ALG_OPDRACHTGEVER'] == 'WSRL') &  # regels filteren op een bepaalde waarde
    (import_data['ANA_TXT_CONSOLIDATIE_TYPE_REKEN'] == 'OC') &  # regels filteren op een bepaalde waarde
    (import_data['ALG_REGIO'] == 'Zuidelijke Waaldijk') &  # regels filteren op een bepaalde waarde
    (
        (import_data['PV_NAAM'] == 'TXT_klei_17.5_21_ondiep') 
    )
]

# Basisplot (maak markers klein)
fig4 = px.scatter(
    filtered_data4,
    y='TXT_SS_T_15%',
    x='ANA_TXT_MAX_VERTICALE_CONSOLIDATIE_SPANNING',
    color='TXT_SS_VOLUMEGEWICHT_DRG',
    title='Su tabel bij 15% rek',
    labels={
        'TXT_SS_MONSTERNIVEAU': 'Monsterniveau [NAP m]',
        'TXT_SS_VOLUMEGEWICHT_DRG': 'Droog volumegewicht [kN/m3]',
        'PV_NAAM': 'PV naam'
    },
    template='plotly_white',
    opacity=0.8,
    hover_data=['ALG__REGEL']
)

# Maak basismarkers kleiner (zorg dat highlight eromheen past)
fig4.update_traces(selector=dict(mode='markers'), marker=dict(size=6))

# Maak highlight subset
highlight = filtered_data4[
    (filtered_data4['TXT_SS_VOLUMEGEWICHT_DRG'] > 15) &
    filtered_data4['ANA_TXT_MAX_VERTICALE_CONSOLIDATIE_SPANNING'].notna() &
    filtered_data4['TXT_SS_T_15%'].notna()
].copy()


# Voeg highlight trace toe (zorg dat deze bovenop komt: use layer='above')
if not highlight.empty:
    fig4.add_trace(
        go.Scatter(
            x=highlight['ANA_TXT_MAX_VERTICALE_CONSOLIDATIE_SPANNING'],
            y=highlight['TXT_SS_T_15%'],
            mode='markers',
            marker=dict(
                size=14,
                color='rgba(255,0,0,0)',    # transparante vulling
                line=dict(color='red', width=3),
                symbol='circle'
            ),
            showlegend=False,
            hoverinfo='skip'
        ),
        secondary_y=False
    )

# Forceer highlight bovenop door update_layout (plotly tekent later toegevoegde traces boven)
fig4.update_layout(legend=dict(itemsizing='constant'))
fig4.update_xaxes(range=[0, None])
fig4.update_yaxes(range=[0, None])

fig4.show()

In [89]:
# S-ratio - Droog volumegewicht

import plotly.express as px

import_data = dbase.dbase_df

filtered_data2 = import_data[
    (import_data['ALG_OPDRACHTGEVER'] == 'WSRL') & #regels filteren op een bepaalde waarde
    (import_data['ANA_TXT_CONSOLIDATIE_TYPE_REKEN'] == 'NC') & #regels filteren op een bepaalde waarde
    (~import_data['PV_NAAM'].fillna('').str.contains('afgekeurd|uitbijter|onbetrouwbaar|telaag|DSS|ondiep', case=False, na=False)) #regels uitsluiten
]

# bereken S-ratio
filtered_data2 = filtered_data2.copy()
filtered_data2['S_NC_15%REK'] = (
    filtered_data2['TXT_SS_T_15%'] /
    filtered_data2['ANA_TXT_MAX_VERTICALE_CONSOLIDATIE_SPANNING'] 
    )

fig2 = px.scatter(
    filtered_data2,
    y='S_NC_15%REK',
    x='TXT_SS_VOLUMEGEWICHT_DRG',
    color='PV_NAAM',
    title='S-ratio Droog volumegewicht',
    labels={
        'S_NC_15%REK': 'NC schuifsterkteratio S 15% rek]',
        'TXT_SS_VOLUMEGEWICHT_DRG': 'Droog volumegewicht [kN/m3]',
        'PV_NAAM': 'PV naam'
    },
    template='plotly_white',
    opacity=0.8,
    hover_data=['ALG__REGEL']  # voeg label uit DB toe aan tooltip
)

fig2.show()


In [124]:
## Figuur opslaan

if import_dropdown.value == 'Proevenverzamelintool 5.0':
    export_dir = str(import_path.parent)
else:
    try:
        selected = getattr(export_dirchooser, "selected_path", None) or getattr(export_dirchooser, "selected", None)
        if selected:
            export_dir = str(selected)
        else:
            export_dir = str(getattr(export_dirchooser, "current_path", export_dirchooser.path))
    except Exception as e:
        raise RuntimeError("Kon exportmap niet ophalen uit export_dirchooser: {}".format(e))

naam_plaatje = 'extra_plaatje'

export_path = Path(export_dir)
output_html = export_path / f"{naam_plaatje}.html"

fig3.write_html(str(output_html))
print(f"HTML figuur opgeslagen in: {output_html}")

HTML figuur opgeslagen in: C:\python_cursus\PVtool2025_github\PV-tool\extra_plaatje.html
